# 05 — Group Relative Policy Optimization (GRPO) and Reinforcement Learning with Verifiable Rewards (RLVR)

## What GRPO and RLVR mean

**Group Relative Policy Optimization (GRPO)** updates a response-generating model using the relative rewards of several sampled answers to the same prompt. A **policy** is that response-generating model.

**Reinforcement Learning with Verifiable Rewards (RLVR)** uses rewards from checks such as a math-answer comparator, a parser, or a test suite. RLVR describes the source of the signal; GRPO describes an optimization approach. They can be combined, but neither implies the other.

**What problem does this solve?** If a task has an affordable, reliable checker, the policy can generate candidate answers without a human writing every demonstration. GRPO derives a baseline from a response group instead of requiring a separate learned value model in this implementation. A checker can still be incomplete or exploitable. Passing a formatting test does not prove factual correctness.


## NovaBot: generate a group and verify it

**Fictional explanatory input:**

~~~json
{"prompt":"How many modes does NovaBot have? Reply with one integer.","answer":"3"}
~~~

Suppose one prompt produces four candidates:

| Candidate | Exact-match reward |
|---|---:|
| 3 | 1 |
| 5 | 0 |
| 3 | 1 |
| three | 0 |

The last answer expresses the right number but violates this deliberately strict checker. A numeric or semantic checker could behave differently. Define what success means before training.

A **reward** is a numerical score; **reward variance** measures spread in those scores. An **advantage** expresses a result relative to a baseline. **On-policy sampling** collects responses from the current policy rather than only using a fixed historical response dataset. The table is hand-constructed, not a set of actual tiny-model rollouts.


## How the group-relative objective works

For one prompt, let $r_i$ be reward for candidate $i$, $G$ the group size, $\mu$ the group mean, and $s$ its population standard deviation. This notebook uses
$$
\mu=\frac{1}{G}\sum_{i=1}^{G}r_i,\qquad
s=\sqrt{\frac{1}{G}\sum_{i=1}^{G}(r_i-\mu)^2},\qquad
A_i=\frac{r_i-\mu}{\max(s,10^{-6})}.
$$
$A_i$ is the normalized advantage. The tiny positive denominator bound prevents division by zero.

For rewards [1,0,1,0], $\mu=0.5$, $s=0.5$, and advantages are [1,-1,1,-1]. For [1,1,1,1], all numerators are zero: every advantage is zero. Normalization cannot invent a ranking among tied answers.

For completion token $t$ of candidate $i$, let $\rho_{it}=\exp(\ell^{\mathrm{new}}_{it}-\ell^{\mathrm{old}}_{it})$ be its new/old probability ratio, with $\ell$ denoting a token log-probability. Let $M_{it}$ be the valid-completion mask and $\epsilon=0.2$ the clip width. The lesson minimizes
$$
L=-\frac{1}{G}\sum_i
\frac{\sum_t M_{it}\min\{\rho_{it}A_i,\operatorname{clip}(\rho_{it},1-\epsilon,1+\epsilon)A_i\}}
{\sum_t M_{it}}.
$$
Across several prompts, it also averages the prompt groups. The displayed experiment sets the reference-penalty coefficient beta to zero, isolating this reward term.

With ratio 1.3 and advantage 1, the clipped surrogate is min(1.3,1.2)=1.2. A batch's mean loss can be zero while gradients remain nonzero, because different sampled responses have different probability derivatives.


## Connection to this notebook

group_advantages() checks finite scores in [0,1], computes group means/stds, and demonstrates the all-equal case. sample_completions() generates candidates in prompt groups. The mask excludes padding; old_logp and advantages are detached from gradient computation. new_logp carries gradients into the policy.

The cells unit-check exact-match and numeric rewards. The actual random-weight rollout experiment uses a **length reward** to expose visible score variation. It does not claim to teach answer correctness. The NovaBot exact-answer checker illustrates a different reward choice.

The policy is trained with **Low-Rank Adaptation (LoRA)**, or its quantized-base variant **QLoRA** for the local profile. The separate Trainer comparison can use different normalization conventions; compare settings before expecting identical scalar losses. Reward mean/std, nonempty completion rate and held-out task success measure different properties.


## Common confusions and quick check

Do not normalize all prompts together as if they shared one comparison group. Do not add arbitrary noise to tied rewards and describe it as verified evidence.

1. Does a high length reward establish that a NovaBot answer is correct?
2. What happens to the relative learning signal if every answer in a group receives zero?

<details>
<summary>Answers</summary>

1. No. It establishes only the property encoded by that length checker.
2. All advantages are zero. Inspect task difficulty, checker behavior and sampling; collect informative groups rather than manufacturing preferences.

</details>


## Before running the experiment

**Learning goals:** sample response groups, validate reward functions, normalize group advantages, and optimize a masked policy objective.

Run top to bottom in a fresh kernel. The tiny profile uses real Qwen classes with random weights. These are mechanics experiments, not quality benchmarks. [Course index](README.md) · [Flow diagrams](../docs/EXECUTION_AND_DATA_FLOW.md)

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Local experiment parameters


In [ ]:
LESSON = "05"
# Parameters: change these before running the notebook from top to bottom.
import csv
import json
import math
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
DEVICE = torch.device("cpu" if MODE == "tiny_cpu" else "cuda")
if MODE == "local_pretrained" and not torch.cuda.is_available():
    raise RuntimeError(
        "The local_pretrained teaching profile requires a CUDA PyTorch installation."
    )
DTYPE = torch.float32 if MODE == "tiny_cpu" else torch.bfloat16
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])

## Load weights and preprocessing

The checkpoint fixes the tokenizer and chat template. A reference or teacher must use a compatible vocabulary. It is frozen but still consumes memory.


In [ ]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "attn_implementation": "eager" if MODE == "tiny_cpu" else "sdpa",
}
# Real 2B teaching runs default to QLoRA. CPU fixtures use ordinary LoRA.
USE_QLORA = MODE == "local_pretrained" and LESSON not in {"00a", "00b", "04"}
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [ ]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.


In [ ]:
def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Template-provided generation masks are preferred. They include assistant EOS.
    template = tokenizer.chat_template or ""
    if "generation" in template:
        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

## Select parameters that may change

LoRA freezes the pretrained matrices and adds low-rank updates. Its zero-initialized B matrices mean some A gradients may be zero on the first step; at least one adapter must change. The frozen vision backbone is excluded. Set `TUNING` to `full` or `selective` to compare on the tiny model; full tuning of a real model needs substantially more memory.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# This is the same choice represented by tuning.strategy in a framework YAML.
TUNING = "lora"
if USE_QLORA and TUNING != "lora":
    raise ValueError("Full/selective tuning requires reloading with USE_QLORA=False.")
if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()
if TUNING == "lora":
    model = get_peft_model(
        model,
        LoraConfig(
            r=4 if MODE == "tiny_cpu" else 16,
            lora_alpha=8 if MODE == "tiny_cpu" else 32,
            target_modules="all-linear",
            exclude_modules=r".*(?:visual|vision).*",
            task_type="CAUSAL_LM",
            lora_dropout=0.0,
        ),
    )
elif TUNING == "selective":
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("norm" in name or "lm_head" in name) and "visual" not in name
elif TUNING != "full":
    raise ValueError(TUNING)
trainable = [p for p in model.parameters() if p.requires_grad]
print(parameter_report(model))
# A small parameter sample avoids copying a 2B/4B model just to audit updates.
before = {name: p.detach().flatten()[:32].cpu().clone() for name, p in model.named_parameters()}
frozen_names = {name for name, p in model.named_parameters() if not p.requires_grad}

## Establish a baseline

Use `eval()` plus `no_grad()` for measurement, and switch back to `train()` for updates. We aggregate causal loss by supervised token count. Greedy decoding makes the before/after and reload comparisons reproducible on the same device.


In [ ]:
def to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def precision_context():
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()


def validation_loss(current_model, rows, collator=collate_text):
    current_model.eval()
    weighted_loss, target_count = 0.0, 0
    with torch.no_grad(), precision_context():
        for row in rows:
            encoded = to_device(collator([row]))
            count = int((encoded["labels"][:, 1:] != -100).sum())
            loss = current_model(**encoded, use_cache=False).loss
            weighted_loss += float(loss) * count
            target_count += count
    return weighted_loss / max(target_count, 1)


def generate_answer(current_model, prompt="What is the color of sky ?"):
    current_model.eval()
    text = render([{"role": "user", "content": prompt}], generation=True)
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    inputs.pop("token_type_ids", None)
    with torch.no_grad(), precision_context():
        tokens = current_model.generate(
            **to_device(dict(inputs)),
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokens[:, inputs["input_ids"].shape[1] :].cpu()


baseline_loss = validation_loss(model, splits["validation"])
baseline_answer = generate_answer(model)
print(
    {
        "baseline_validation_loss": baseline_loss,
        "baseline_answer": tokenizer.decode(baseline_answer[0], skip_special_tokens=True),
    }
)

## Reward design comes before optimization

Exact-match and numeric rewards verify answers; format and length rewards verify different properties. Check malformed output, missing answers and score ranges. A length objective is easy to exploit and is used here only to make random tiny-model rollouts produce inspectable variation. It does not teach correctness.


In [ ]:
from finetunelab.rewards.builtin import exact_match_reward, length_reward, numeric_reward

assert exact_match_reward(["2", "3"], answer=["2", "2"]) == [1.0, 0.0]
assert numeric_reward(["Answer: 2", "not a number"], answer=["2", "2"]) == [1.0, 0.0]
try:
    exact_match_reward(["2"])
except ValueError as error:
    print("Expected missing-answer diagnostic:", error)
else:
    raise AssertionError("A verification reward must require a reference answer.")


def group_advantages(rewards):
    if not torch.isfinite(rewards).all() or not ((rewards >= 0) & (rewards <= 1)).all():
        raise ValueError("Expected finite rewards in [0, 1].")
    std = rewards.std(dim=1, unbiased=False, keepdim=True)
    return (rewards - rewards.mean(dim=1, keepdim=True)) / std.clamp_min(1e-6), std.squeeze(-1)


equal_advantages, _ = group_advantages(torch.ones(2, 4))
assert torch.equal(equal_advantages, torch.zeros_like(equal_advantages))
print("All-equal reward groups contribute zero advantage; do not invent ranking signals.")

## Sample groups and compute advantages

GRPO compares several completions of the **same** prompt. Group normalization must not mix unrelated prompts. This notebook uses population standard deviation explicitly; compare that convention with your selected Trainer configuration before expecting numerical equivalence.


In [ ]:
def sample_completions(current_model, prompts, count=1):
    # Left padding keeps each prompt's final real token at the generation boundary.
    previous_padding = tokenizer.padding_side
    tokenizer.padding_side = "left"
    rendered = [render([{"role": "user", "content": p}], generation=True) for p in prompts]
    encoded = tokenizer(rendered, add_special_tokens=False, padding=True, return_tensors="pt")
    tokenizer.padding_side = previous_padding
    encoded.pop("token_type_ids", None)
    encoded = to_device(dict(encoded))
    current_model.eval()
    with torch.no_grad(), precision_context():
        sequences = current_model.generate(
            **encoded,
            do_sample=True,
            temperature=1.0,
            top_k=0,
            top_p=1.0,
            max_new_tokens=4 if MODE == "tiny_cpu" else 32,
            num_return_sequences=count,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    width = encoded["input_ids"].shape[1]
    completions = sequences[:, width:]
    # Include the first EOS, exclude subsequent padding even when PAD == EOS.
    eos = completions.eq(tokenizer.eos_token_id)
    prior_eos = eos.cumsum(-1) - eos.long()
    completion_mask = prior_eos.eq(0)
    if tokenizer.pad_token_id != tokenizer.eos_token_id:
        completion_mask &= completions.ne(tokenizer.pad_token_id)
    attention = torch.cat(
        [
            encoded["attention_mask"].repeat_interleave(count, 0),
            completion_mask.long(),
        ],
        dim=1,
    )
    return sequences, attention, width, completion_mask


def completion_logprobs(current_model, sequences, attention, width):
    logits = (
        current_model(input_ids=sequences, attention_mask=attention, use_cache=False)
        .logits[:, width - 1 : -1]
        .float()
    )
    targets = sequences[:, width:]
    return logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1), logits


GROUP_SIZE = 4
prompts = [r["prompt"] for r in splits["train"][:2]]
sequences, attention, width, mask = sample_completions(model, prompts, count=GROUP_SIZE)
completion_texts = tokenizer.batch_decode(sequences[:, width:], skip_special_tokens=True)
rewards = torch.tensor(length_reward(completion_texts, target_length=32), device=DEVICE)
rewards = rewards.reshape(len(prompts), GROUP_SIZE)
advantages, reward_std = group_advantages(rewards)
display(list(zip(completion_texts, rewards.flatten().tolist(), strict=False)))
assert advantages.abs().sum() > 0, (
    "All groups have equal reward; revise task/reward or collect more samples."
)
with torch.no_grad():
    old_logp, _ = completion_logprobs(model, sequences, attention, width)
advantages = advantages.flatten().detach()

## One token-masked group-relative update

This experiment chooses `beta=0` to isolate the reward advantage term; no reference model is required. With a KL penalty, load a frozen reference and add the configured estimator over the same completion mask. Old probabilities and advantages are detached, while new probabilities carry gradients.


In [ ]:
optimizer = torch.optim.AdamW(trainable, lr=2e-3 if MODE == "tiny_cpu" else 2e-4)
optimizer.zero_grad(set_to_none=True)
model.train()
new_logp, _ = completion_logprobs(model, sequences, attention, width)
ratio = (new_logp - old_logp).exp()
unclipped = ratio * advantages[:, None]
clipped = ratio.clamp(0.8, 1.2) * advantages[:, None]
per_token_loss = -torch.minimum(unclipped, clipped)
loss = ((per_token_loss * mask).sum(-1) / mask.sum(-1).clamp_min(1)).mean()
assert torch.isfinite(loss)
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in trainable if p.grad is not None)
torch.nn.utils.clip_grad_norm_(trainable, 1.0)
optimizer.step()
print(
    {
        "grpo_loss": float(loss.detach()),
        "reward_mean": float(rewards.mean()),
        "reward_std": reward_std.tolist(),
        "completion_validity": sum(bool(t.strip()) for t in completion_texts)
        / len(completion_texts),
    }
)
model.eval()
with torch.no_grad():
    heldout_sequences, _, heldout_width, _ = sample_completions(
        model,
        [r["prompt"] for r in splits["validation"]],
        count=GROUP_SIZE,
    )
    heldout_text = tokenizer.batch_decode(
        heldout_sequences[:, heldout_width:], skip_special_tokens=True
    )
print("Held-out reward:", length_reward(heldout_text, target_length=32))

## Inspect updates and held-out behavior

A finite loss and a changed adapter prove an update occurred, not that a model became useful. Inspect validation loss and example generations together. Keep the test set out of hyperparameter selection.


In [ ]:
changed = []
for name, parameter in model.named_parameters():
    same = torch.equal(before[name], parameter.detach().flatten()[:32].cpu())
    if name in frozen_names:
        assert same, f"Frozen parameter changed: {name}"
    elif not same:
        changed.append(name)
assert changed, "No trainable parameter sample changed."
after_loss = validation_loss(model, splits["validation"])
after_answer = generate_answer(model)
test_loss = validation_loss(model, splits["test"])
print(
    {
        "baseline_loss": baseline_loss,
        "after_loss": after_loss,
        "test_loss": test_loss,
        "changed_parameter_samples": changed[:5],
    }
)
print("Before:", tokenizer.decode(baseline_answer[0], skip_special_tokens=True))
print("After: ", tokenizer.decode(after_answer[0], skip_special_tokens=True))
# Do not assert that generalization improves after one synthetic update.
assert math.isfinite(after_loss) and math.isfinite(test_loss)

## Save, reload, and verify

`save_pretrained()` saves inference artifacts; it does not save the optimizer or training position. A PEFT artifact needs its original base. This cell creates a separate model object and compares generated token IDs. Lesson 08 covers exact resume and merging.


In [ ]:
from peft import PeftModel

artifact = OUTPUT / "final"
model.save_pretrained(artifact, safe_serialization=True)
processor.save_pretrained(artifact)
expected_tokens = generate_answer(model)
reloaded_processor = AutoProcessor.from_pretrained(artifact, local_files_only=True)
assert reloaded_processor.tokenizer.get_vocab() == tokenizer.get_vocab()
assert reloaded_processor.chat_template == processor.chat_template
processor = reloaded_processor
tokenizer = processor.tokenizer
# Reload independently, rather than reusing the trained Python object.
if (artifact / "adapter_config.json").exists():
    reload_base = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
    if not USE_QLORA:
        reload_base.to(DEVICE)
    reloaded = PeftModel.from_pretrained(reload_base, artifact, local_files_only=True)
else:
    reloaded = AutoModelForImageTextToText.from_pretrained(artifact, **load_kwargs)
    if not USE_QLORA:
        reloaded.to(DEVICE)
actual_tokens = generate_answer(reloaded)
assert torch.equal(expected_tokens, actual_tokens), "Greedy outputs changed after reload."
report = {
    "mode": MODE,
    "base_checkpoint": str(MODEL_PATH),
    "artifact": str(artifact),
    "baseline_validation_loss": baseline_loss,
    "validation_loss": after_loss,
    "test_loss": test_loss,
    "reload_tokens_equal": True,
}
(OUTPUT / "lesson_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
del reloaded
if "reload_base" in globals():
    del reload_base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Interpretation, common failures, and exercises

- If loss is NaN, inspect the number of supervised targets, precision and learning rate before adding steps.
- If every label is `-100`, repair the template/mask; an empty objective cannot teach anything.
- If frozen parameters change, inspect the trainable parameter report and optimizer parameter list.
- If tiny generations look meaningless, that is expected from random initialization and a tiny vocabulary.

**Exercises:** (1) Print which token predicts the first answer token. (2) Compare full/selective/LoRA parameter counts. (3) Change one training answer, rerun from the same seed, and inspect held-out loss. (4) Explain why saving an adapter is not enough to resume AdamW.

**Expected result:** finite objective values, some expected trainable weights changed, frozen weights unchanged, and identical greedy tokens after reload. Record actual values in `lesson_report.json`; no fixed quality threshold is asserted.


## Connect this calculation to FineTuneLab

The recipe builds the production trainer from the same canonical records. The CPU profile executes a separate one-step comparison. The real multi-model comparison is shown explicitly but is deferred to a fresh process to avoid keeping two experiments in GPU memory.


In [ ]:
METHOD = "grpo"
framework_rows = [{"prompt": r["prompt"]} for r in splits["train"][:2]]
# Map the visible experiment back to the framework boundary.
# Start a separate short run from the same base, not from the mutated notebook model.
from datasets import Dataset

from finetunelab.config import RECIPE_ADAPTER
from finetunelab.data import validate_dataset
from finetunelab.recipes import build_trainer

recipe = {
    "method": METHOD,
    "model": {
        "name_or_path": str(MODEL_PATH),
        "local_files_only": True,
        "dtype": "float32" if MODE == "tiny_cpu" else "bfloat16",
    },
    "data": {"source": str(OUTPUT / "train.jsonl"), "max_length": 128},
    "tuning": {"strategy": "lora", "lora_rank": 4, "lora_alpha": 8},
    "training": {
        "output_dir": str(OUTPUT / "framework"),
        "max_steps": 1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "gradient_checkpointing": False,
        "bf16": MODE != "tiny_cpu",
        "tf32": False,
        "logging_steps": 1,
        "save_steps": 1,
        "report_to": ["none"],
    },
    "generation": {"max_new_tokens": 4},
}
if METHOD == "grpo":
    recipe.update({"num_generations": 2, "beta": 0.0, "rewards": [{"name": "length"}]})
config = RECIPE_ADAPTER.validate_python(recipe)
framework_dataset = Dataset.from_list(framework_rows)
print(validate_dataset(framework_dataset, config))
# True in CPU acceptance tests. Real multi-model runs need sufficient GPU memory.
RUN_FRAMEWORK_COMPARISON = MODE == "tiny_cpu"
if RUN_FRAMEWORK_COMPARISON:
    trainer = build_trainer(config, framework_dataset)
    framework_result = trainer.train()
    trainer.save_model(str(OUTPUT / "framework" / "final"))
    print(framework_result.metrics)
    assert all(
        math.isfinite(float(v))
        for v in framework_result.metrics.values()
        if isinstance(v, (int, float))
    )
    del trainer
else:
    print(
        "Configuration validated. In a fresh process run "
        "build_trainer(config, framework_dataset).train()."
    )